[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Toan-ten/Arxiv-Project-Group10/blob/fix%2Fm2-nocite-checkpoint-migration/notebooks/m2/m2_nocite_recovery.ipynb)

# M2 — Recovery sau bản vá nocite (CPU-only)

Notebook **riêng**, không thay notebook team. Không dùng Run all. Chạy từng ô theo thứ tự, đọc preflight trước khi xác nhận ghi. Dừng notebook cũ; chỉ một writer cho mỗi campaign.

| Bước | Tác động Google Drive |
|---|---|
| Setup | Mount và đọc identity; code/dependencies/config ở `/content` |
| Preflight / dry-run | **Chỉ đọc** campaign/archives; giải nén vào scratch `/content`, không parse |
| Migrate | Ghi campaign mới và verified markers; campaign cũ chỉ đọc |
| Package legacy | Sao chép Parquet và lineage vào campaign mới; không parse |
| Pilot 0115 | Ghi checkpoint/Parquet mới, chỉ parse paper cần sửa thuộc **0115** |
| Audit pilot / audit | **Chỉ đọc**, không tạo lock hay ghi artifacts |
| Continue | Ghi kết quả các cohort sau pilot, tối đa BATCH_SIZE/lượt |

Giữ old campaign để audit lineage. Không tự xóa lock sau hard stop. Không crawl/API/HF upload. Dataset recovery có provenance riêng từng publication; không dùng làm final benchmark.

In [ ]:
# 1. Cấu hình — giữ nguyên các giá trị này sau khi tạo recovery campaign.
OLD_CAMPAIGN = "/content/drive/MyDrive/ArxivGroup10/24120232/m2-exports/24120232/0101-0150-072ee384efa2-3ffc7291afbd"
RECOVERY_ROOT = ""  # Trống: tự dùng ArxivGroup10/<owner>/m2-recovery từ identity cũ.
PILOT_COHORT = 115
BATCH_SIZE = 5  # Có thể đổi sau Setup; không đổi campaign identity.


In [ ]:
# 2. Setup — không migration hoặc parse. Clone đúng release và cài Arrow CPU.
import os, sys, json, subprocess
from pathlib import Path
from google.colab import drive
RELEASE_COMMIT = "e6869ff26e36b7122128b7ff165a2bd3da9003b4"
PARSER_FINGERPRINT = "4e88c7fb0d3463c4c6815b5afff3f64066815068dd4c34db1e92528966d60b70"
RELEASE_BRANCH = "fix/m2-nocite-checkpoint-migration"
if not (3, 11) <= sys.version_info[:2] < (3, 15):
    raise RuntimeError("Cần Python 3.11–3.14; chưa kiểm chứng phiên bản khác.")
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Pipeline này không dùng GPU.
drive.mount("/content/drive")
CODE_DIR = Path("/content/group10-m2-recovery") / RELEASE_COMMIT[:12]
if not CODE_DIR.exists():
    CODE_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--no-checkout", "--single-branch", "--branch", RELEASE_BRANCH,
                    "https://github.com/Toan-ten/Arxiv-Project-Group10.git", str(CODE_DIR)], check=True)
    subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "--detach", RELEASE_COMMIT], check=True)
if subprocess.check_output(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"], text=True).strip() != RELEASE_COMMIT:
    raise RuntimeError("Checkout không đúng release; không reset/ghi đè code đang có.")
if subprocess.check_output(["git", "-C", str(CODE_DIR), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Checkout có thay đổi; dùng runtime sạch, không tự xóa.")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pyarrow==26.0.1"], check=True)
SCRATCH = "/content/m2-nocite-recovery-scratch"
LOCAL_CONFIG = Path("/content/m2-nocite-recovery-config.json")
_SETUP_SELECTION = (OLD_CAMPAIGN, RECOVERY_ROOT, PILOT_COHORT)
PREFLIGHT = None
print("Setup hoàn tất. Runtime:", RELEASE_COMMIT, "— chưa ghi campaign, chưa parse.")


In [ ]:
# 3. Công cụ điều khiển — chỉ định nghĩa hàm, chưa chạy nghiệp vụ.
def current_config():
    if (OLD_CAMPAIGN, RECOVERY_ROOT, PILOT_COHORT) != _SETUP_SELECTION:
        raise RuntimeError("Đã đổi campaign/pilot; restart runtime và Setup lại để tránh dùng nhầm identity.")
    old = json.loads((Path(OLD_CAMPAIGN) / "campaign.json").read_text(encoding="utf-8-sig"))
    owner = old["owner"]
    return dict(repo=str(CODE_DIR), student_id=owner, start_cohort=old["scope"]["start"],
        end_cohort=old["scope"]["end"], batch_size=BATCH_SIZE,
        input_drive_dir=old["archives"], scratch=SCRATCH, hf_repo_id=old["hf_repo_id"],
        old_campaign=OLD_CAMPAIGN,
        recovery_root=RECOVERY_ROOT or f"/content/drive/MyDrive/ArxivGroup10/{owner}/m2-recovery",
        parser_commit=RELEASE_COMMIT, parser_fingerprint=PARSER_FINGERPRINT,
        allow_uncommitted=False, pilot_cohort=PILOT_COHORT)


def command(action, *, approved=False, capture=False):
    config = current_config()
    if approved:
        if not PREFLIGHT or not PREFLIGHT["ready"]:
            raise RuntimeError("Chạy lại Preflight/dry-run và đọc kết quả trước khi ghi Drive.")
        config["approved_preflight_token"] = PREFLIGHT["preflight_token"]
    LOCAL_CONFIG.write_text(json.dumps(config), encoding="utf8")  # Chỉ /content.
    args = [sys.executable, "-X", "utf8", "-B", "-m", "scripts.m2_colab_recovery",
            "--config", str(LOCAL_CONFIG), "--action", action]
    if capture:
        result = subprocess.run(args, cwd=CODE_DIR, text=True, capture_output=True)
        if result.returncode:
            print(result.stdout); print(result.stderr)
            raise RuntimeError("Preflight/audit dừng an toàn; không ghi đè hoặc xóa evidence.")
        return json.loads(result.stdout.strip().splitlines()[-1])
    subprocess.run(args, cwd=CODE_DIR, check=True)


def show_preflight(action="preflight"):
    global PREFLIGHT
    PREFLIGHT = None
    report = command(action, capture=True)
    print("Campaign mới:", report["recovery_campaign"])
    print("Có thể migrate:", report["can_migrate"])
    print("Packaged ở campaign cũ:", report["old_packaged"])
    print("Packaged ở recovery:", report["recovery_packaged"])
    for row in report["cohorts"]:
        if row["status"] != "PENDING_NOT_INSPECTED":
            print(json.dumps(row, ensure_ascii=False))
    print("Pending chưa kiểm tra source:", sum(r["status"] == "PENDING_NOT_INSPECTED" for r in report["cohorts"]))
    if not report["ready"]:
        raise RuntimeError("Chưa đủ điều kiện ghi Drive; xem errors/writer_locks.")
    PREFLIGHT = report
    return report


def write_action(action):
    if not PREFLIGHT or not PREFLIGHT["ready"]:
        raise RuntimeError("Cần preflight PASS trước.")
    expected = f"{action.upper()} {PILOT_COHORT:04d}" if action in {"pilot", "continue"} else action.upper()
    answer = input(f"Bước này GHI campaign mới. Nhập {expected!r} để chạy: ").strip()
    if answer != expected:
        print("Đã dừng; chưa chạy thao tác ghi.")
        return
    command(action, approved=True)


### 4. Preflight và dry-run — chỉ đọc Drive
Chạy cả hai ô, kiểm tra `ready`, đúng recovery path, **0101–0114 MIGRATABLE**, 0101–0105 packaged ở campaign cũ; không tự suy 0106–0114 đã packaged. 0115 phải liệt kê paper cần reparse và checkpoint bị loại. File thiếu/sai checksum/provenance → dừng; không sửa JSON để ép PASS. 0116–0150 hiện là PENDING_NOT_INSPECTED; source sẽ được kiểm trước khi chạy từng cohort.

In [ ]:
show_preflight("preflight");


In [ ]:
show_preflight("dry-run");


### 5. Migration — GHI campaign mới, không parse
Chỉ sau khi đọc preflight. Bước này tạo verified markers/provenance tham chiếu artifacts cũ đã kiểm chứng. Không xóa hoặc đổi campaign cũ. Nếu lỗi, giữ nguyên thư mục mới dở dang để review; không tự rollback bằng việc xóa dữ liệu.

In [ ]:
write_action("migrate")


### 6. Package phần legacy — GHI campaign mới, không parse
Chạy ô này lặp lại tới khi `recovery_packaged` có đủ 0101–0114. Mỗi lần tối đa BATCH_SIZE, sao chép Parquet và provenance vào recovery. Package cũ giữ nguyên. Có thể đặt BATCH_SIZE=50 ở ô cấu hình để đóng gói 14 cohort trong một lượt, không cần Setup lại.

In [ ]:
write_action("package")
show_preflight("preflight");


### 7. Pilot — chỉ cohort 0115, GHI kết quả mới
Helper chặn nếu các cohort trước pilot chưa migrate/package đủ. BATCH_SIZE không mở rộng lượt pilot. Chỉ paper lỗi/không có cache hợp lệ được parse, giữ mọi versions của từng publication. Checkpoint lỗi cũ không được seed.

In [ ]:
write_action("pilot")


### 8. Audit pilot — chỉ đọc Drive
Kiểm own-version citation evidence, schema, checksum, payload và version coverage. Xem `excluded_legacy_checkpoints`/`parser_origins` trong recovery `verified/cohort_0115_corpus.json`. Không tiếp tục nếu paper bị lỗi nocite trước đây còn mất version thành công. Parse_error ở paper khác phải được ghi trung thực, không được coi là dữ liệu đã parse đủ.

In [ ]:
command("audit-pilot")


### 9. Continue — GHI, chỉ sau pilot verified/packaged/audit
Mỗi lần chạy tối đa BATCH_SIZE cohort sau 0115 chưa packaged, tiếp tục đến 0150 theo scope cũ. Có thể đổi BATCH_SIZE trong ô cấu hình; không đổi old campaign, recovery root hay pilot. Không tự chạy hết dải trong một vòng lặp. Sau runtime mới, chạy lại ô 1–4; giữ đúng release/paths, preflight lại rồi Continue. Nếu còn legacy chưa migrate/package hoặc pilot chưa hoàn tất thì dùng lại các ô tương ứng.

In [ ]:
write_action("continue")


### 10. Audit và rollback
Audit toàn bộ recovery đã verified là thao tác chỉ đọc. Chỉ xem package có `READY.json`, `checksums.json`, `coverage.json` và batch marker tương ứng là completed. Coverage ghi riêng provenance legacy/new, không tuyên bố tất cả dùng cùng parser.

Nếu migration lỗi: dừng writer, giữ logs/partial recovery, **không xóa/sửa old campaign**. Nếu Stop bình thường, chạy lại cùng release/config; nếu hard reset để lại `.export-writer.lock` hoặc `.writer.lock`, xác minh không còn process ghi rồi review lock riêng, không tự xóa checkpoint. Old campaign vẫn dùng được để đọc/audit completed; không tiếp tục 0115 bằng parser cũ đang lỗi. Không upload đồng thời package cũ/mới trùng cohort.

Chưa xác minh Google Drive FUSE và checkpoint thực tế bằng notebook này trong lúc chuẩn bị release. Preflight trên runtime của bạn là bước nghiệm thu dữ liệu thật, không được bỏ qua.

In [ ]:
command("audit")
